# The full benchmark on RAW RV — 10 models + HAR-RV × 10 datasets × h = 1, 5, 22

The same grid as `colab_benchmark.ipynb`, on the **variance scale** instead of
`ln(RV)`: every model forecasts the h-day forward mean of RV directly, HAR-RV
is fitted on the same rows without `--log`, and everything is scored there.

Nothing in the repository changes. The modelling scale is a property of the
**anchors** — a tuned command line carrying `--log` makes a cell `ln_RV`, its
absence makes it `raw_RV` — and `run_benchmark.py` reads it off them, refuses
to mix the two inside one sweep, and fits HAR-RV on whichever scale it finds.
The shipped anchors in `tuning/ProjectC_tuning` all carry `--log`, so a raw
sweep needs anchors of its own. That is what makes this two stages:

**A. Re-tune on raw RV (§5–§7)** — the same Optuna protocol the log anchors
got: the same search spaces, the same 50 trials per model, the same 3 repeats
per trial, the same `seq_len 96 / pred_len 1 / --aggregate_mean` on EUR/USD at
h = 1. One flag differs, `--raw`. It writes `<Model>_best.json` files whose
command lines have no `--log`.

**B. The sweep (§9–§11)** — the benchmark, pointed at those anchors.

Re-tuning rather than reusing the log winners is the point of the exercise.
`ln(RV)` is close to Gaussian and raw RV is right-skewed with a heavy tail, so
the loss surface is a different one: the learning rate, batch size and schedule
that won on it are not the ones that win here, and a model handed the wrong
step size would be reported as unsuited to the raw scale when it was only
mistuned for it. §8 has a transplant cell if you want the other experiment —
same hyper-parameters, raw target — but it answers a different question.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

**This does not finish in one session.** Stage A is up to 10 × 50 × 3 = **1 500
trainings**, stage B is **3 000** (300 cells × 10 seeds) plus 10 HAR-RV fits; a
Colab session is capped at ~12 h. Both stages write to Google Drive and both
**resume** — reconnect, re-run steps 1–4, then re-run whichever stage you were
in. Optuna tops each study up to its target instead of restarting it, and a
benchmark cell already on Drive is skipped. §12 shows how far along you are;
§13's tables work on a half-finished sweep.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > T4 GPU. 4500 trainings on CPU is not realistic.'

## 2. Mount Drive

The Optuna database, the raw anchors, the forecasts, the tables and the loss
matrices go here so they survive a disconnect — that is what makes both stages
resumable across sessions.

Both directories are **separate from the log run's**, deliberately. Anchors
carry the scale, so a directory holding both kinds would make
`run_benchmark.py` refuse the sweep ("one scale per sweep"), and a results
directory holding both would produce blocks the aggregation reports as mixed
and skips. Keeping them apart is also what lets the two sweeps be compared
afterwards (§15).

Checkpoints do **not** go to Drive: they are rewritten every improving epoch,
and on a Drive mount that would dominate the runtime. They are deleted after
each trial and after each cell anyway.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ANCHOR_DIR   = '/content/drive/MyDrive/ProjectC_tuning_raw'      # stage A writes here
RESULTS_DIR  = '/content/drive/MyDrive/ProjectC_benchmark_raw'   # stage B writes here
CKPT_DIR     = '/content/_ckpt'                                  # local disk, deleted per trial/cell
STUDY_PREFIX = 'rv_raw'   # Optuna study names. Distinct from the log study's 'rv'
                          # prefix, so a raw run can never top up a log study
                          # even if the two directories get crossed.

import os
os.makedirs(ANCHOR_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('anchors ->', ANCHOR_DIR)
print('results ->', RESULTS_DIR)

## 3. Clone the repository

`Mr0022/ProjectC` is public, so this needs no credentials. Re-running the cell
in a later session updates an existing clone instead of failing.

Switch `BRANCH` to `'main'` once this work is merged.

In [ ]:
import os, subprocess

BRANCH   = 'claude/benchmark-raw-rv-7ef3kb'
REPO_URL = 'https://github.com/Mr0022/ProjectC.git'
REPO_DIR = '/content/ProjectC'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)   # the code resolves models/ and data/ relatively
print(subprocess.run(['git', 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install the dependencies Colab is missing

Colab already ships torch, numpy, pandas, scikit-learn, statsmodels and
matplotlib. These are the extras this repo needs — `optuna` for stage A,
`fast_pytorch_kmeans` for AdaWaveNet, `reformer-pytorch`/`local-attention`
because `layers/SelfAttention_Family.py` imports them at module load.

In [ ]:
!pip install -q optuna einops PyWavelets fast_pytorch_kmeans reformer-pytorch local-attention psutil

import torch, optuna
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| optuna', optuna.__version__)

## 5. Sanity check the raw search (~1 minute)

Two trials of the cheapest model on the raw target, written to local disk so it
cannot be mistaken for part of the real study. The `python -u run.py ...` line
printed at the end is the thing to look at: it must contain `--aggregate_mean`
and **must not** contain `--log`. That absence is the entire difference between
this notebook and the log one.

One consequence of it is worth knowing even though the `tail` below hides it:
`--aggregate_mean` implies `--drop_nonpositive` in `run.py` (it prints
`Non-positive targets will be dropped` at the top of every trial), so the raw
run drops the same zero-RV non-trading days the log run drops and HAR-RV drops
unconditionally. The two families stay fitted on identical rows — five of the
forex series carry two such days each.

In [ ]:
!python tuning/optuna_tune.py --model DLinear --raw --n_trials 2 --train_epochs 3 \
    --out_dir /content/_smoketune --checkpoint_dir $CKPT_DIR 2>&1 | tail -12

## 6. Stage A — the raw search, 50 trials per model, 3 repeats each

Identical to `tuning/colab_tune.ipynb`'s step 6 except for `--raw`: same
`tuning/search_spaces.py` ranges, same 50-trial target, same 3 seeds per trial
scored by their mean validation loss, same EUR/USD h = 1 study. Equal search
effort for every architecture is what makes the benchmark table a comparison of
architectures rather than of tuning budgets, and re-using the log study's
protocol verbatim is what keeps the raw benchmark comparable to the log one.

One model per call, so progress is saved after each. `N_TRIALS` is a **target
per model**, not a batch size: the driver counts what the database already
holds and runs only the remainder, so this cell is safe to re-run as often as
you like, and a model already at 50 is skipped. Editing `MODELS` runs a subset
— the list below is ordered cheapest-first, which is the practical way to get a
usable set of anchors before the expensive models finish.

Rough cost on a T4, the same as the log study: DLinear/FITS ~10–20 minutes
each, PatchTST/TSLANet/iTransformer/ModernTCN/AdaWaveNet/TimeMixer an hour or
more, MSGNet/TimesNet several hours.

Raw RV is heavy-tailed, so more of the top of the learning-rate range diverges
here than it did on `ln(RV)`. A trial that blows up is recorded as a failed or
pruned trial and the search moves on — TPE stops proposing that corner after a
few of them, which is the mechanism working, not a problem to fix.

In [ ]:
MODELS   = ['DLinear', 'FITS', 'TSLANet', 'PatchTST', 'iTransformer', 'ModernTCN',
            'AdaWaveNet', 'TimeMixer', 'MSGNet', 'TimesNet']
N_TRIALS = 50
N_SEEDS  = 3     # repeats per trial; the objective is their mean

import subprocess, time
for model in MODELS:
    print(f'\n{"="*72}\n  {model}  [raw RV]\n{"="*72}', flush=True)
    started = time.time()
    subprocess.run(['python', 'tuning/optuna_tune.py',
                    '--model', model,
                    '--raw',
                    '--n_trials', str(N_TRIALS),
                    '--n_seeds', str(N_SEEDS),
                    '--study_prefix', STUDY_PREFIX,
                    '--out_dir', ANCHOR_DIR,
                    '--checkpoint_dir', CKPT_DIR])
    print(f'{model} finished in {(time.time()-started)/60:.1f} min', flush=True)

## 7. The winners, and the check that they really are raw

The one thing that must hold before stage B: no anchor carries `--log`.
`run_benchmark.py` checks it too and would refuse the sweep, but it costs a
second here and saves a confusing error later.

`val_loss` is the mean best validation loss over the trial's 3 seeds, measured
on the **raw** target. It is not comparable with the log study's number — a
different target, not a rescaling of the same one — so only the ranking inside
this table means anything.

In [ ]:
import glob, json, os
import pandas as pd

ALL_MODELS = ['DLinear', 'PatchTST', 'iTransformer', 'TimesNet', 'MSGNet',
              'TimeMixer', 'FITS', 'TSLANet', 'ModernTCN', 'AdaWaveNet']

rows = []
for path in sorted(glob.glob(os.path.join(ANCHOR_DIR, '*_best.json'))):
    b = json.load(open(path))
    rows.append({'model': b['model'],
                 'scale': 'ln_RV' if '--log' in b['command'].split() else 'raw_RV',
                 'val_loss': b['best_val_loss'],
                 'val_std': b.get('best_val_loss_std'),   # spread over the trial's seeds
                 'trials': b['n_complete'],
                 'minutes': round(b['seconds'] / 60, 1)})

if not rows:
    print('no anchors in', ANCHOR_DIR, '— run step 6 first')
else:
    df = pd.DataFrame(rows).sort_values('val_loss').reset_index(drop=True)
    df.to_csv(os.path.join(ANCHOR_DIR, 'best_configs.csv'), index=False)
    display(df)

    missing = [m for m in ALL_MODELS if m not in set(df['model'])]
    print(f'{len(df)}/10 models tuned'
          + (f'; not yet: {", ".join(missing)} (stage B skips them)' if missing else ''))
    wrong = df.loc[df['scale'] != 'raw_RV', 'model'].tolist()
    print('SCALE CHECK: ' + ('every anchor is raw_RV' if not wrong
                             else f'!! these still carry --log: {wrong}'))

In [ ]:
# the exact command line that reproduces each winner
for path in sorted(glob.glob(os.path.join(ANCHOR_DIR, '*_best.json'))):
    b = json.load(open(path))
    val = b.get('best_val_loss')          # None for a transplanted anchor (see §8)
    head = f'# {b["model"]}' + (f'  (val {val:.6g})' if val is not None else '')
    print(f'{head}\n{b["command"]}\n')

## 8. Optional — reuse the log winners instead of re-tuning

**A different experiment from §6.** This takes the log study's winning
hyper-parameters and re-points them at the raw target by dropping `--log` from
the command line. It holds the architecture and the optimiser settings fixed
and changes only the scale, which isolates the effect of the scale — but it
hands every model a learning rate, batch size and schedule chosen on the
`ln(RV)` loss surface, so a model that does badly under it may be mistuned
rather than unsuited to raw RV. That is the trade: §6 costs 1 500 trainings and
gives each architecture its own best shot on the raw scale; this costs nothing
and gives a clean one-variable comparison against the log sweep.

It writes to its own directory and touches neither §6's anchors nor the shipped
log ones. To use it, uncomment, run it, then set `ANCHOR_DIR` to `DST` and
`RESULTS_DIR` to a fresh directory before running stage B.

`best_val_loss` is cleared on the way through: the number in the log study's
file is a validation loss on `ln(RV)` and does not describe this anchor. It is
kept under `best_val_loss_ln_study`, so the provenance stays readable and
`metrics.csv`'s `anchor_val_loss` column comes out empty rather than wrong.

In [ ]:
# import glob, json, os
#
# SRC = 'tuning/ProjectC_tuning'                                  # the shipped log anchors
# DST = '/content/drive/MyDrive/ProjectC_tuning_raw_transplant'
# os.makedirs(DST, exist_ok=True)
#
# for path in sorted(glob.glob(os.path.join(SRC, '*_best.json'))):
#     b = json.load(open(path))
#     b['command'] = ' '.join(t for t in b['command'].split() if t != '--log')
#     b['best_val_loss_ln_study'] = b.get('best_val_loss')
#     b['best_val_loss'] = None      # the ln(RV) number is not this anchor's
#     b['transplanted_from'] = path
#     with open(os.path.join(DST, os.path.basename(path)), 'w') as fh:
#         json.dump(b, fh, indent=2)
#     print(f'{b["model"]:<14} -> {DST}')
#
# print('\nnow set  ANCHOR_DIR = DST  and give RESULTS_DIR a fresh directory')

## 9. Validate every raw configuration (~1 minute, CPU)

Builds all 10 models at all 3 horizons from the raw anchors and pushes one
batch through each, then prints how many forecasts each dataset holds per
horizon. A configuration the architecture rejects should surface here, not at
hour six of the sweep.

`--scale raw_RV` is the guard on the whole notebook: the flag has to agree with
what the anchors say, so pointing `--anchor_dir` at a log study by mistake
fails here in a second with a message naming both scales.

In [ ]:
!python orchestrate/run_benchmark.py --validate \
    --anchor_dir $ANCHOR_DIR --scale raw_RV

## 10. Smoke test the sweep (~1 minute)

Two cheap models plus the baseline on one dataset at one horizon, one seed, 5
epochs — written to local disk, so a smoke cell can never be mistaken for a
real one by the resume logic. If it ends with a summary table headed
`h = 1  [raw_RV]`, the environment is wired up correctly — the scale in that
header is read back off the stored cells, not off the flag.

Expect the HAR-RV line to mention `har_rv_h01_fitted.csv`, not
`har_rv_log_h01_fitted.csv`: the baseline is fitted raw here too.

In [ ]:
!python -u orchestrate/run_benchmark.py \
    --datasets EURUSD --horizons 1 --models DLinear FITS HAR-RV --itr 1 --quick \
    --anchor_dir $ANCHOR_DIR --scale raw_RV \
    --results_dir /content/_smoke_raw --checkpoint_dir $CKPT_DIR 2>&1 | tail -25

## 11. Stage B — the sweep

3 000 trainings (300 cells × 10 seeds) plus 10 HAR-RV fits. One subprocess per
cell, so an OOM or a CUDA fault costs one cell rather than the run; each cell
writes its forecasts the moment it finishes, and cells already on Drive are
skipped — which is what makes this cell safe to re-run after every disconnect.

Ordered dataset → horizon → model, so an interrupted run leaves **complete**
(dataset, horizon) blocks behind — the unit the MCS is defined over.

`EXTRA` runs a subset, which is the practical way to do this over several
sessions:

* `['--assets', 'crypto']` or `['--datasets', 'EURUSD', 'AUDUSD']`
* `['--horizons', '1']`
* `['--itr', '3']` — 3 repeats per cell instead of 10, roughly a third of the cost
* `['--models', 'FITS', 'DLinear', 'TSLANet']` — the cheap ones first

A model with no raw anchor yet is reported as skipped rather than silently
dropped, so stage B can be started while the expensive stage-A studies are
still running, and the missing models filled in on a later pass.

In [ ]:
import subprocess, time

EXTRA = []          # e.g. ['--assets', 'crypto'] or ['--itr', '3']

started = time.time()
subprocess.run(['python', '-u', 'orchestrate/run_benchmark.py',
                '--anchor_dir', ANCHOR_DIR,
                '--scale', 'raw_RV',
                '--results_dir', RESULTS_DIR,
                '--checkpoint_dir', CKPT_DIR] + EXTRA)
print(f'\nthis session ran for {(time.time() - started) / 3600:.2f} h')

## 12. How far along is it?

The planner is the authority on what is left — it counts the files on Drive the
same way the sweep does. The grid below shows completed cells per dataset and
horizon (110 = 11 models × 10 seeds, with HAR-RV counting once per horizon).

In [ ]:
!python orchestrate/run_benchmark.py --results_dir $RESULTS_DIR \
    --anchor_dir $ANCHOR_DIR --scale raw_RV --dry_run 2>&1 | head -12

In [ ]:
import glob, os
import pandas as pd

paths = glob.glob(os.path.join(RESULTS_DIR, 'runs', '*', 'h*', '*.npz'))
if not paths:
    print('nothing on Drive yet')
else:
    done = pd.DataFrame([{'dataset': p.split(os.sep)[-3],
                          'horizon': p.split(os.sep)[-2]} for p in paths])
    print(f'{len(done)} cell(s) on Drive')
    display(done.value_counts().unstack(fill_value=0))

## 13. The tables

`aggregate_results.py` re-scores whatever is on Drive — no retraining — so this
also works on a sweep that is still running. It says which blocks are still
short of models.

A raw sweep produces **no `MSE_ln` / `MAE_ln`**: those losses are the squared
and absolute error in `ln(RV)`, and the log of a forecast that can be negative
does not exist. The columns are kept in `metrics.csv` but stay empty, and no
`pivot_MSE_ln_*` file is written. `QLIKE`, `MSE_RV` and `MAE_RV` are the three
metrics this sweep reports, and they are exactly the three that are directly
comparable with the log sweep's (§15).

In [ ]:
!python orchestrate/aggregate_results.py --results_dir $RESULTS_DIR

In [ ]:
import os
import pandas as pd

TABLES = os.path.join(RESULTS_DIR, 'tables')

# mean +/- std over the seeds, per dataset and horizon
display(pd.read_csv(os.path.join(TABLES, 'metrics_mean.csv')).head(20))

# models x datasets, one metric, one horizon -- the shape a paper table has
for metric in ('QLIKE', 'MSE_RV'):
    for h in (1, 5, 22):
        path = os.path.join(TABLES, f'pivot_{metric}_h{h:02d}.csv')
        if os.path.exists(path):
            print(f'\n{metric}, h = {h}')
            display(pd.read_csv(path, index_col=0))

## 14. The raw-scale health check — non-positive forecasts

This cell has no counterpart in the log notebook, because under `--log` it is
identically zero by construction: a log-scale forecast back-transforms to
`exp(.) > 0`. On the raw scale an unconstrained head — and HAR-RV's levels OLS
— can issue a **negative variance**, and the two losses handle it differently:

* `MSE_RV` / `MAE_RV` score against `max(forecast, 0)`, since a negative
  variance is not admissible and zero is the best feasible prediction the model
  could have issued;
* `QLIKE` is infinite at zero, so those forecasts are floored at
  `1e-4 × mean training RV` instead.

Both rules come from `HAR-RV_RUN.PY`'s raw branch, so the baseline and the deep
models are treated identically. But a model with a large share of floored
forecasts is being scored partly on the floor rather than on what it predicted,
and its QLIKE is a statement about the floor as much as about the model. That
belongs in the caption of any raw table, which is why it is counted here.

In [ ]:
import os
import pandas as pd

m = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'metrics.csv'))
total, floored = int(m['n_obs'].sum()), int(m['n_floored'].sum())
print(f'{floored} of {total} stored forecasts are <= 0 '
      f'({100 * floored / max(total, 1):.3f}%)')

by_model = m.groupby('model')[['n_floored', 'n_obs']].sum()
by_model['pct'] = (100 * by_model['n_floored'] / by_model['n_obs']).round(3)
display(by_model.sort_values('n_floored', ascending=False))

# target_dev is the other thing worth a glance: how far each cell's own actuals
# sit from Y^(h) rebuilt from the CSV, relative to the level of RV on the raw
# scale. ~1e-7 is the deep models' float32 round trip through the loader's
# scaler; anything larger means a cell was scored on the wrong rows.
print(f"\nlargest target deviation: {m['target_dev'].max():.2e}")

## 15. Optional — raw against log, side by side

Only the variance-scale metrics are shared between the two sweeps, and that is
not a limitation of the tables: `exp()` of the log target is the raw target
exactly, at every horizon, so `QLIKE`, `MSE_RV` and `MAE_RV` are the same
function of the same actuals in both. `MSE_ln` has no raw counterpart at all.

Point `LOG_RESULTS` at the `ln(RV)` sweep's results directory. Negative
`delta_*` means the raw sweep loses less.

In [ ]:
import os
import pandas as pd

LOG_RESULTS = '/content/drive/MyDrive/ProjectC_benchmark'   # the ln(RV) sweep

log_path = os.path.join(LOG_RESULTS, 'tables', 'metrics_mean.csv')
if not os.path.exists(log_path):
    print('no log-scale tables at', log_path, '— skip this cell')
else:
    keys = ['dataset', 'horizon', 'model']
    shared = ['QLIKE', 'MSE_RV', 'MAE_RV']
    raw = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'metrics_mean.csv'))
    ln = pd.read_csv(log_path)
    both = raw[keys + shared].merge(ln[keys + shared], on=keys,
                                    suffixes=('_raw', '_ln'))
    for metric in shared:
        both[f'delta_{metric}'] = both[f'{metric}_raw'] - both[f'{metric}_ln']

    print(f'{len(both)} (dataset, horizon, model) cell(s) scored on both scales')
    display(both.head(20))

    # how often each scale wins, per horizon
    wins = (both.assign(raw_wins=both['delta_QLIKE'] < 0)
                .groupby('horizon')['raw_wins'].agg(['sum', 'count']))
    wins.columns = ['raw lower QLIKE', 'cells']
    display(wins)

## 16. The DM / MCS inputs

`losses/<dataset>_h<hh>__<loss>__seed<S>.csv` is date-indexed, one column per
model, one row per forecast. A raw sweep writes three of them — `qlike`,
`se_rv`, `ae_rv` — where the log sweep writes five; `se_ln` and `ae_ln` are
simply absent, for the reason in §13. The mean of a column **is** the matching
cell in `metrics.csv`.

**Run the tests on the `__seedmean` files**: one series per model — the
forecast its ten repeats average to — so you get one DM statistic and one MCS,
rather than ten that cannot be pooled. In raw mode the modelling-scale and
variance-scale averages coincide, so the ensemble is the plain mean of the ten
forecasts.

At h > 1 the target windows of consecutive rows overlap by h−1 days, so the
loss differential is autocorrelated *by construction*: the DM long-run variance
needs a HAC estimator with at least h−1 lags, and the MCS block bootstrap needs
a block length that respects the same overlap.

In [ ]:
import glob, os
import pandas as pd

LOSSES = os.path.join(RESULTS_DIR, 'losses')
pick = sorted(glob.glob(os.path.join(LOSSES, '*__qlike__seedmean.csv')))
if not pick:
    print('no seedmean matrices yet — they appear once a block has >1 seed')
else:
    path = pick[0]
    print(os.path.basename(path))
    L = pd.read_csv(path, index_col=0, parse_dates=True)
    print(L.shape, '(forecasts x models)')
    display(L.head())

    # HAR-RV minus each model, row by row: positive = the model loses less.
    # rsub with axis=0 broadcasts down the dates; a plain Series - DataFrame
    # would align on the column labels and give an all-NaN frame.
    if 'HAR-RV' not in L.columns:
        print('no HAR-RV column in this block — nothing to compare against')
    else:
        d = L.drop(columns='HAR-RV').rsub(L['HAR-RV'], axis=0)
        display(d.mean().sort_values(ascending=False)
                 .to_frame('mean QLIKE saved vs HAR-RV'))

## 17. Download the results (optional)

They are already on Drive. This is for pulling the small files — the raw
anchors, the tables and the loss matrices, not the 3 000 `.npz` — onto your
laptop in one archive.

In [ ]:
import glob, os, zipfile

out = '/content/ProjectC_benchmark_raw_tables.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ('tables', 'forecasts', 'losses'):
        for root, _, names in os.walk(os.path.join(RESULTS_DIR, sub)):
            for name in names:
                path = os.path.join(root, name)
                z.write(path, os.path.relpath(path, RESULTS_DIR))
    manifest = os.path.join(RESULTS_DIR, 'manifest.json')
    if os.path.exists(manifest):
        z.write(manifest, 'manifest.json')
    # the anchors this sweep was run from -- without them the table is not
    # reproducible, and they are a few kB
    for path in sorted(glob.glob(os.path.join(ANCHOR_DIR, '*_best.json'))):
        z.write(path, os.path.join('anchors_raw', os.path.basename(path)))

print(out, round(os.path.getsize(out) / 1e6, 1), 'MB')

from google.colab import files
files.download(out)

## Notes

* **What actually differs from the log benchmark.** The anchors, and nothing
  else. No file in the repository is edited, no flag of `run_benchmark.py`
  changes meaning; `--log` is absent from the tuned command lines, and the
  target, the losses, HAR-RV's own invocation and the QLIKE floor all follow
  from that. `--scale raw_RV` on every call is a check that they did.
* **Resuming.** Stage A: the studies live in `optuna.db` under `ANCHOR_DIR`;
  re-running §6 tops each up to `N_TRIALS` and skips models already there.
  Stage B: a cell whose `.npz` is on Drive is skipped, and a failed cell is
  recorded in `failures.csv` and skipped too — `['--retry_failed']` in `EXTRA`
  re-runs those.
* **Non-positive forecasts are the raw scale's characteristic failure mode**
  and are not an error: §14 counts them, `metrics.csv` carries `n_floored` per
  cell, and the aggregation prints a warning when any exist. Under `--log`
  there are none, so the two sweeps are not on equal footing here — a raw model
  is spending some of its QLIKE on the floor.
* **Which metrics exist.** `QLIKE`, `MSE_RV`, `MAE_RV`. `MSE_ln` and `MAE_ln`
  are empty columns and no `pivot_MSE_ln_*` is written. The three that exist are
  comparable with the log sweep's, because `exp()` of the log target is the raw
  target exactly at every horizon.
* **Hyper-parameters** are the raw Optuna winners for **EUR/USD at h = 1**,
  applied to every dataset and horizon — the same transfer the log benchmark
  makes, and it belongs in the caption of any table built from these results.
  Per-dataset anchors are a drop-in: put
  `<ANCHOR_DIR>/<dataset>/<Model>_best.json` in place and that dataset picks it
  up automatically.
* **The training budget is not the tuning budget.** The studies search at
  `--train_epochs 30 --patience 7`; the sweep runs every cell at 50 / 10,
  because `train_epochs` and `patience` are flags the orchestrator owns. That is
  the same mismatch the log benchmark has, kept deliberately so the two are
  comparable — but note that a `--lradj cosine` anchor divides by
  `train_epochs` to shape its schedule, so for those cells the cap is not merely
  a stopping point.
* **Drive is slow for many small files.** Stage B writes one `.npz` per cell
  (3 000 of them) and the aggregation a few thousand CSVs. That is fine, but a
  §13 run takes a minute or two once the grid is full.
* **Cost.** FITS, DLinear and TSLANet are seconds per cell; TimesNet and MSGNet
  dominate both stages. Plan on several sessions, or start stage B with
  `['--itr', '3']`.
* Details, file formats and the two `exp/` changes this work made:
  `orchestrate/README.md`; the search spaces: `tuning/README.md` and
  `tuning/search_spaces.py`.